In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from fredapi import Fred
import warnings

from plotly.data import stocks

warnings.filterwarnings("ignore")

df = pd.read_csv("data.csv", sep=";", decimal=",")
df = df.rename(columns={
    "Column1": "Date",
    "Column2": "SPX",
    "Column3": "S5SFTW",
    "Column4": "S5PHRM",
    "Column5": "S5CPGS",
    "Column6": "S5ENRSX",
    "Column7": "S5FDBT",
    "Column8": "S5TECH",
    "Column9": "S5RETL",
    "Column10": "S5BANKX",
    "Column11": "S5HCES",
    "Column12": "S5DIVF",
    "Column13": "S5UTILX",
    "Column14": "S5MEDA",
    "Column15": "S5REAL",
    "Column16": "S5TELSX",
    "Column17": "S5MATRX",
    "Column18": "S5INSU",
    "Column19": "S5FDSR",
    "Column20": "S5HOUS",
    "Column21": "S5SSEQX",
    "Column22": "S5TRAN",
    "Column23": "S5HOTR",
    "Column24": "S5CODU",
    "Column25": "S5AUCO",
    "Column26": "S5COMS",
})
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

In [67]:
def GetReturn(df,date,lookback):
    date=pd.to_datetime(date)
    if date not in df["Date"].values:#add breaker if windows not in df
        raise ValueError("Date not in dataframe")
    returns_df = df[["Date","S5SFTW","S5PHRM","S5CPGS","S5ENRSX","S5FDBT","S5TECH","S5RETL","S5BANKX","S5HCES","S5DIVF","S5UTILX","S5MEDA","S5REAL","S5TELSX","S5MATRX","S5INSU","S5FDSR","S5HOUS","S5SSEQX","S5TRAN","S5HOTR","S5CODU","S5AUCO","S5COMS"]].copy()

    date_index = returns_df.index[returns_df["Date"] == date][0]
    returns_df=returns_df[(returns_df.index<=date_index) & (returns_df.index>=date_index-lookback) ]
    returns_df.drop(columns="Date",inplace=True)

    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)
    #print(returns_df.std().mean()) #verification if std is around 1% daily

    return returns_df

#return a df of size (lookback, number of sectors) with log returns


def GetReturnSPX(df,date,lookback):
    date=pd.to_datetime(date)
    if date not in df["Date"].values:#add breaker if windows not in df
        raise ValueError("Date not in dataframe")
    returns_df = df[["Date","SPX"]].copy()

    date_list=returns_df.drop(columns="Date")
    date_index = returns_df.index[returns_df["Date"] == date][0]

    returns_df=returns_df[(returns_df.index<=date_index) & (returns_df.index>=date_index-lookback) ]
    returns_df.drop(columns="Date",inplace=True)

    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)
    #print(returns_df.std().mean()) #verification if std is around 1% daily

    return returns_df

#return a df of size (lookback, 1) with log returns of SPX

In [68]:
def GetSigma(df,date,lookback):

    returns_df=GetReturn(df,date,lookback=lookback)
    #covariance matric from returns_df
    sigma_windowed=returns_df.cov()

    return sigma_windowed

from sklearn.covariance import LedoitWolf,OAS
import pandas as pd

def get_shrunk_covariance(df,date,lookback):

    returns=GetReturn(df,date,lookback)

    lw = OAS()
    lw.fit(returns)
    shrunk_cov = lw.covariance_

    delta = lw.shrinkage_
    if isinstance(returns, pd.DataFrame):
        shrunk_cov = pd.DataFrame(
            shrunk_cov,
            index=returns.columns,
            columns=returns.columns
        )


    return shrunk_cov


def getSigmaModified(df,date,lookback,listofbanneddays,periodison=False):

    date=pd.to_datetime(date)
    if date not in df["Date"].values:#add breaker if windows not in df
        raise ValueError("Date not in dataframe")
    returns_df = df[["Date","S5SFTW","S5PHRM","S5CPGS","S5ENRSX","S5FDBT","S5TECH","S5RETL","S5BANKX","S5HCES","S5DIVF","S5UTILX","S5MEDA","S5REAL","S5TELSX","S5MATRX","S5INSU","S5FDSR","S5HOUS","S5SSEQX","S5TRAN","S5HOTR","S5CODU","S5AUCO","S5COMS"]].copy()

    date_index = returns_df.index[returns_df["Date"] == date][0]
    returns_df=returns_df[(returns_df.index<=date_index) & (returns_df.index>=date_index-lookback)]
    #days selection

    #banned days
    for banned_date in listofbanneddays:
        mask = returns_df["Date"] == banned_date
        if mask.any():
            print("got one :", banned_date)
            returns_df.loc[mask, :] = np.nan

    returns_df.drop(columns="Date",inplace=True)
    returns_df.dropna(inplace=True)

    #calculation of returns
    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)

    #covaraicne matrix using shrinkage
    lw = OAS()
    lw.fit(returns_df)
    shrunk_cov = lw.covariance_

    delta = lw.shrinkage_
    if isinstance(returns_df, pd.DataFrame):
        shrunk_cov = pd.DataFrame(
            shrunk_cov,
            index=returns_df.columns,
            columns=returns_df.columns
        )


    return shrunk_cov

#return a cov matrix of size (number of sectors, number of sectors) we use lookback to have different window sizes

In [69]:
def GetRfDataframe(df):
    fred = Fred(api_key="5c742a53d96bd3085e9199dcdb5af60b")
    riskfree = fred.get_series('DFF')
    # riskfree = fred.get_series('DTB1MO')

    riskfree = riskfree.to_frame(name='FedFunds')
    riskfree.index.name = "Date"
    riskfree = riskfree[riskfree.index >= "2002-01-01"]
    riskfree["FedFunds"]=riskfree["FedFunds"]/100
    list_days_open = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    list_days_full = pd.to_datetime(riskfree.index, dayfirst=True, errors="coerce")

    list_days_open=[pd.to_datetime(date) for date in list_days_open]
    list_days_full=[pd.to_datetime(date) for date in list_days_full]


    list_days_open_pondered=[]
    riskfree_list=[]
    count_list=[]
    timestamp=0
    while timestamp < len(list_days_full)-1:

      if list_days_full[timestamp+1] in list_days_open:
            list_days_open_pondered.append(list_days_full[timestamp])
            riskfree_list.append(riskfree["FedFunds"].loc[list_days_full[timestamp]])
            count_list.append(1)
            timestamp += 1

      else:
          count = 0
          timestampbis = timestamp
          while (timestamp + 1 < len(list_days_full)) and (list_days_full[timestamp + 1] not in list_days_open):
              timestamp += 1
              count += 1

          list_days_open_pondered.append(list_days_full[timestampbis])  # jour de départ
          riskfree_list.append(riskfree["FedFunds"].loc[list_days_full[timestampbis]])
          count_list.append(count+1)
          timestamp += 1

    RfDf=pd.DataFrame({"Date":list_days_open_pondered,"Rf":riskfree_list,"Count":count_list})
    RfDf=RfDf.set_index("Date")
    return RfDf

def GetRiskFree(df,date,lookback,RfDf):
    positionOfStartDate=df.index[df["Date"]==pd.to_datetime(date)][0]-lookback
    #print(positionOfStartDate)
    startDate=pd.to_datetime(df.iloc[positionOfStartDate,0])
    endDate=pd.to_datetime(date)
    RfDf=RfDf[(RfDf.index >= startDate) & (RfDf.index <= endDate )].copy()
    CumulativeRf=[]

    for i in range(len(RfDf)):
      if i==0:
        CumulativeRf.append(pow((1+RfDf["Rf"].iloc[i]),(RfDf["Count"].iloc[i]/360)))
      else:
        CumulativeRf.append(pow((1+RfDf["Rf"].iloc[i]),(RfDf["Count"].iloc[i]/360))*CumulativeRf[i-1])

    RfDf["CumulativeRf"]=CumulativeRf
    RfDf["CumulativeRf"]= RfDf["CumulativeRf"]-1

    return RfDf["CumulativeRf"].iloc[-1]

RfDf=GetRfDataframe(df)

#compute risk free dataframe using API from FRED and get the cumulative risk free rate between two dates in a df

In [70]:
def GetWeight(df,date):
    #for the moment we will use the equal weight
    weight_vector=np.zeros((24,1))
    for i in range(0,24):
        weight_vector[i]=1/24

    return weight_vector

#usual weighting scheme, for the moment equal weight

In [71]:
holderlambda=[]
datesss=[]
def GetLambda(df,date,timeofcalculation,RfDf):
    returns=GetReturn(df,date,timeofcalculation) #daily returns
    weight_vector=GetWeight(df=0,date=0)

    mean_return=np.mean(np.dot(returns,weight_vector))
    mean_annual=(1+mean_return)**252-1 #annualized mean return


    rf_temps=GetRiskFree(df,date,timeofcalculation,RfDf)
    rf_annual=(1+rf_temps)**(252/timeofcalculation)-1 #annualized risk free rate


    Sigma=get_shrunk_covariance(df,date,timeofcalculation)
    Sigma_annual=252*Sigma #annualized covariance matrix
    var = float((weight_vector.T @ Sigma_annual.values @ weight_vector).item())
    lambda_value=(mean_annual - rf_annual)/var


    excess = mean_annual - rf_annual
    sigma2 = var
    sigma  = np.sqrt(var)
    lam    = excess / sigma2
    sharpe = excess / sigma
    #print("Excess:", excess, " Var:", sigma2, " Vol:", sigma, " λ:", lam, " Sharpe:", sharpe)
    datesss.append(date)
    holderlambda.append(lambda_value)
    return lambda_value

#compute the lambda value using the mean return, risk free rate and variance of the portfolio

Lambda=GetLambda(df,"2024-01-11",timeofcalculation=3500,RfDf=RfDf)



In [72]:
#add the Q matrix calculation

def QMatrixCalculation(df,date,lookback,proportion,performerc_daily,dailyperf_market,historical_returns):
    Q=np.zeros((proportion,1))
    factor=1
    for i in range(proportion):
        Q[i,0]=(performerc_daily[i][0]-dailyperf_market)/2


    return Q,historical_returns

In [73]:
def GetPMatrix(df,date, lookback,proportion=3,historical_returns=0):

    AssetColumns=["S5SFTW","S5PHRM","S5CPGS","S5ENRSX","S5FDBT","S5TECH","S5RETL","S5BANKX","S5HCES","S5DIVF","S5UTILX","S5MEDA","S5REAL","S5TELSX","S5MATRX","S5INSU","S5FDSR","S5HOUS","S5SSEQX","S5TRAN","S5HOTR","S5CODU","S5AUCO","S5COMS"]
    bestperformer = []
    performerc = []
    performerc_daily=[]
    returnBestPerformer=[]
    endDateIndex=df.index[df["Date"]==pd.to_datetime(date)][0]
    startDateIndex=df.index[df["Date"]==pd.to_datetime(date)][0]-lookback

    for i in range(2, df.shape[1]):  #loop through asset columns
        performerc.append((((float(df.iloc[endDateIndex, i]) / float(df.iloc[startDateIndex, i]) - 1) * 100), i - 2,df.columns[i])) #pos of best stock in a tuple
        # with its return
        performerc_daily.append(((float(df.iloc[endDateIndex, i]) / float(df.iloc[startDateIndex, i])) ** (1/lookback) - 1, i - 2,df.columns[i])) #daily version


    performerc.sort(reverse=True)
    performerc_daily.sort(reverse=True)
    #print(performerc)
    perfMarket= (float(df.iloc[endDateIndex, 1]) / float(df.iloc[startDateIndex, 1]) - 1) * 100
    dailyperf_market = (float(df.iloc[endDateIndex, 1]) / float(df.iloc[startDateIndex, 1])) ** (1/lookback) - 1






    for i in range(proportion):
        bestperformer.append(performerc_daily[i][1])
        returnBestPerformer.append(performerc_daily[i][0])


    P=np.zeros((proportion,24))
    Q=np.zeros((proportion,1))
    for lineview in range(proportion):
        for i in range(len(AssetColumns)):
            P[lineview,i]=-1/len(AssetColumns)
        P[lineview,bestperformer[lineview]]=1-1/len(AssetColumns)
        sum=0
        for i in range(len(AssetColumns)):
            sum+=P[lineview,i]
    Q,historical_returns=QMatrixCalculation(df,date,lookback,proportion,performerc_daily,dailyperf_market,historical_returns)


    return P, Q, historical_returns

In [74]:
def GetOmega(PMatrix, Sigma, c=0.99):
    #Omega is the uncertainty of the views

    factorC=(1/c-1)
    Omega=factorC*PMatrix@Sigma@np.transpose(PMatrix)

    return Omega



In [75]:
def LinkOmegaTau(Omega, Sigma, P):
    #Link omega to tau
    constant=36

    multiple= np.trace(np.transpose(P) @ np.linalg.inv(Omega) @ P) * constant
    numerator= np.trace(np.linalg.inv(Sigma*252))

    result= numerator / multiple
    return result



In [76]:
def LinkOmegaTau2(Omega, Sigma,P,tau):
    #Link omega to tau
    numerator= np.trace(np.linalg.inv(Sigma*tau))
    denominator= np.trace((np.transpose(P)@np.linalg.inv(Omega)@P))
    result=numerator/denominator

    return result



In [77]:
#residual momentum
def GetFullReturnsForResidual(df):
    returns_df = df[["S5SFTW","S5PHRM","S5CPGS","S5ENRSX","S5FDBT","S5TECH","S5RETL","S5BANKX","S5HCES","S5DIVF","S5UTILX","S5MEDA","S5REAL","S5TELSX","S5MATRX","S5INSU","S5FDSR","S5HOUS","S5SSEQX","S5TRAN","S5HOTR","S5CODU","S5AUCO","S5COMS"]].copy()
    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)
    return returns_df
    #print(returns_df.std().mean()) #verification if std is around 1% daily
GetFullReturnsForResidual(df)

def GetReturnDaily(df,date,lookback):

    date=pd.to_datetime(date)
    if date not in df["Date"].values:#add breaker if windows not in df
        raise ValueError("Date not in dataframe")
    returns_df = df[["Date","S5SFTW","S5PHRM","S5CPGS","S5ENRSX","S5FDBT","S5TECH","S5RETL","S5BANKX","S5HCES","S5DIVF","S5UTILX","S5MEDA","S5REAL","S5TELSX","S5MATRX","S5INSU","S5FDSR","S5HOUS","S5SSEQX","S5TRAN","S5HOTR","S5CODU","S5AUCO","S5COMS"]].copy()
    date_index = returns_df.index[returns_df["Date"] == date][0]
    returns_df=returns_df[(returns_df.index<=date_index) & (returns_df.index>=date_index-lookback)]
    returns_df.drop(columns="Date",inplace=True)
    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)
    #print(returns_df.std().mean()) #verification if std is around 1% daily
    return returns_df


def GetFullReturnsForResidualSPXDaily(df,date,lookback):

    date=pd.to_datetime(date)
    if date not in df["Date"].values:#add breaker if windows not in df
        raise ValueError("Date not in dataframe")
    returns_df = df[["Date","SPX"]].copy()

    date_list=returns_df.drop(columns="Date")
    date_index = returns_df.index[returns_df["Date"] == date][0]

    returns_df=returns_df[(returns_df.index<=date_index) & (returns_df.index>=date_index-lookback) ]
    returns_df.drop(columns="Date",inplace=True)
    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)
    #print(returns_df.std().mean()) #verification if std is around 1% daily

    return returns_df

In [78]:
def GetReturnMonthly(df,date,lookback):
    date=pd.to_datetime(date)
    if date not in df["Date"].values:#add breaker if windows not in df
        raise ValueError("Date not in dataframe")
    returns_df = df[["Date","S5SFTW","S5PHRM","S5CPGS","S5ENRSX","S5FDBT","S5TECH","S5RETL","S5BANKX","S5HCES","S5DIVF","S5UTILX","S5MEDA","S5REAL","S5TELSX","S5MATRX","S5INSU","S5FDSR","S5HOUS","S5SSEQX","S5TRAN","S5HOTR","S5CODU","S5AUCO","S5COMS"]].copy()
    date_index = returns_df.index[returns_df["Date"] == date][0]
    returns_df=returns_df[(returns_df.index<=date_index) & (returns_df.index>=date_index-lookback)]
    returns_df.drop(columns="Date",inplace=True)
    returns_df=returns_df.iloc[::21].copy()
    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)
    #print(returns_df.std().mean()) #verification if std is around 1% daily
    return returns_df

#return a df of size (lookback, number of sectors) with log returns


def GetReturnSPXMonthly(df,date,lookback):
    date=pd.to_datetime(date)
    if date not in df["Date"].values:#add breaker if windows not in df
        raise ValueError("Date not in dataframe")
    returns_df = df[["Date","SPX"]].copy()

    date_list=returns_df.drop(columns="Date")
    date_index = returns_df.index[returns_df["Date"] == date][0]

    returns_df=returns_df[(returns_df.index<=date_index) & (returns_df.index>=date_index-lookback) ]
    returns_df.drop(columns="Date",inplace=True)
    returns_df=returns_df.iloc[::21].copy()
    returns_df = np.log(returns_df/ returns_df.shift(1))
    returns_df.dropna(inplace=True)
    #print(returns_df.std().mean()) #verification if std is around 1% daily

    return returns_df

#return a df of size (lookback, 1) with log returns of SPX

In [79]:
import statsmodels.api as sm
def Residual(dfretassets,dfretspx):
    rfmonthly=(0.02/12)
    y=dfretassets-rfmonthly
    x=dfretspx-rfmonthly

    x_with_const = sm.add_constant(x)

    model = sm.OLS(y, x_with_const)
    results = model.fit()

    params = results.params
    alpha = params.loc["const"]             # scalaire si y est Series, Series si y est DataFrame
    beta  = params.loc["SPX"]               # idem

    residus = results.resid

    return alpha, beta, residus

In [99]:
def GetPandQUsingRedisual(df,lookbackdate,lookback,proportion,historical_returns):
    #36 months data
    alpha,beta,residus=Residual(GetReturnMonthly(df,lookbackdate,12),GetReturnSPXMonthly(df,lookbackdate,12))
    beta=beta.tolist()
    alpha=alpha.tolist()
    # get the n last residual for the score computation
    a,b,residus=Residual(GetReturnDaily(df,lookbackdate,lookback),GetFullReturnsForResidualSPXDaily(df,lookbackdate,lookback))
    backforscore=3*22 #in months
    listofperf=[]
    for i in range(residus.shape[1]):
        extract=[residus.iloc[residus.shape[0]-j-1,i] for j in range(0,backforscore)]
        zscore=np.sum(extract)/np.std(extract)
        listofperf.append((alpha[i],beta[i],zscore,i))

    listofperf=sorted(listofperf,key=lambda x:x[2],reverse=True)

    AssetColumns=["S5SFTW","S5PHRM","S5CPGS","S5ENRSX","S5FDBT","S5TECH","S5RETL","S5BANKX","S5HCES","S5DIVF","S5UTILX","S5MEDA","S5REAL","S5TELSX","S5MATRX","S5INSU","S5FDSR","S5HOUS","S5SSEQX","S5TRAN","S5HOTR","S5CODU","S5AUCO","S5COMS"]


    P=np.zeros((proportion,24))
    for lineview in range(proportion):
        for i in range(len(AssetColumns)):
            P[lineview,i]=-1/len(AssetColumns)
        P[lineview,listofperf[lineview][3]]=1-1/len(AssetColumns)



    #Q prediction !

    #compute the monthly returns
    dailyret_market=GetFullReturnsForResidualSPXDaily(df,lookbackdate,lookback)
    meandaily=np.mean(dailyret_market)

    Q=np.zeros((proportion,1))
    for stuff in range(proportion):
        prediction=listofperf[stuff][0]+listofperf[stuff][1]*meandaily
        excess=prediction-meandaily
        Q[stuff,0]=excess



    historical_returns=[]
    return P,Q,historical_returns

#GetPandQUsingRedisual(df,"2018-05-11",21*36)



In [100]:
StackLambda=[]
datesss2=[]
def BlackAndLittermanModel(backtestStartDate, rebalancingFrequency, lookbackPeriod, df,RfDf,confidence=0.75,proportion=4,tau=0.025,Lambda=3,historical_returns=0,modifiedlambda=0):
    #implement the full backtest of the black and litterman model

    #---------
    #PARAMETERS
    #---------
    datetoremove=pd.to_datetime("2018-04-06") #add date to remove
    listofbanneddays=[]
    Sigma=get_shrunk_covariance(df,backtestStartDate,lookback=60) #using 720 days to have better sigma of 2 years
    Sigma=getSigmaModified(df,backtestStartDate,lookback=60,listofbanneddays=listofbanneddays) #using 720 days to have better sigma of 2 years


    PMatrix,Q,historical_returns= GetPandQUsingRedisual(df,backtestStartDate, lookback=lookbackPeriod,proportion=proportion,historical_returns=historical_returns)
    Omega=GetOmega(PMatrix, Sigma, c=confidence)
    rf=GetRiskFree(df,backtestStartDate,lookbackPeriod,RfDf)
    weights = GetWeight(df, backtestStartDate)
    weights = np.array(weights).reshape(-1, 1)

    changingLambda=False
    if changingLambda==True:
        Lambda=3+0.05*GetLambda(df,backtestStartDate,timeofcalculation=60,RfDf=RfDf)
    else :
        Lambda=3

    uimplied = Lambda * (Sigma @ weights) + rf
    #BL formula
    #tau=OmegaLinked
    StackLambda.append(Lambda)
    datesss2.append(backtestStartDate)




    optimizedReturn=(np.linalg.inv(np.linalg.inv(tau*Sigma)+np.transpose(PMatrix)@np.linalg.inv(Omega)@PMatrix)) @ (np.linalg.inv(tau*Sigma)@uimplied+np.transpose(PMatrix)@np.linalg.inv(Omega)@Q)
    LambdaMarkowitz=3

    #MarkowitzAllocation
    WeightBL=np.linalg.inv(Sigma)@(optimizedReturn-rf)/LambdaMarkowitz
    WeightRF=1-np.sum(WeightBL)
    #if not np.isclose(float(np.sum(WeightBL)), 1.0, atol=1e-6):
        #print(np.sum(WeightBL))
        #raise ValueError("Weights do not sum to 1, please investigate.")

    return WeightBL,WeightRF,historical_returns


BlackAndLittermanModel("2018-05-11", rebalancingFrequency=3, lookbackPeriod=180, df=df,RfDf=RfDf)


ValueError: zero-size array to reduction operation maximum which has no identity

In [94]:
from rich.console import Console
from rich.panel import Panel
from tqdm import tqdm

console = Console()

#BACK TESTER
dfbacktest=df.copy()
dfbacktest["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
dfbacktest["MonthIndex"] = dfbacktest["Date"].dt.to_period("M")

df_length = dfbacktest.shape[1] - 2  # bcs of date and spx
last_rebalance = dfbacktest.loc[0, "Date"]  # première date
month_count = 0

# 🎨 AFFICHAGE STYLÉ (sans prompts)
hold = 1
hist = 0
proportion = 4
Lambda=3
tau=0.025
confidence=0.75

console.print(Panel.fit(
    "[bold cyan]📊 PORTFOLIO BACKTESTER[/bold cyan]\n"
    "[dim]Black-Litterman Model[/dim]",
    border_style="cyan"
))

console.print(f"\n[yellow]⚙️  Configuration :[/yellow]")
console.print(f"   • Hold period: [cyan]{hold}[/cyan] mois")
console.print(f"   • Historique: [cyan]{hist}[/cyan] mois")
console.print(f"   • Proportion: [cyan]{proportion}[/cyan]")
console.print(f"   • Lambda: [cyan]{Lambda:.4f}[/cyan]")
console.print(f"   • Confiance: [cyan]{confidence}[/cyan]")
console.print(f"   • Taux: [cyan]{tau}[/cyan]\n")

console.print("\n[yellow]⏳ Lancement du backtest...[/yellow]\n")

def Backtester(df,hold, hist, proportion,df_toBL, RfDf,confidence2,proportion2,tau2,Lambda2,start,modifiedlambda):
    #new dataframe for stock quantity

    StockQty = df.copy()
    StockQty.drop(columns="MonthIndex", inplace=True)
    historical_returns=[]

    StockQty.loc[:, :] = 0
    #starting data
    MoneyAtStart = 10000000
    month_count=0
    CurrentValue=MoneyAtStart
    spaceindays=0
    #first ligne
    StockQty.loc[start, "Money"] = MoneyAtStart
    StockQty.loc[start, "SPX"] = df.iloc[start, 1]
    StockQty.loc[start, "Date"] = df.iloc[start, 0]
    RiskFreeAmount=0
    #start of the algorithm

    for i in tqdm(range(start,df.shape[0]), desc="Backtesting"):
      StockQty.iloc[i,0]=df.iloc[i,0]
      StockQty.iloc[i,1]=df.iloc[i,1]
      fees=0


      if df.loc[i, "Date"].month != df.loc[i-1, "Date"].month:
        month_count += 1


    # Si on atteint la période voulue
      if i>= hist and spaceindays>21*hold:
        #print(f"🔁 Rebalancement déclenché à la date : {df.loc[i, 'Date'].date()}")
        #print(str(df.iloc[i,0]))

        spaceindays=0

        BLWeight,RiskFreeAmount,historical_returns=BlackAndLittermanModel(str(df.iloc[i,0]),3,3*22,df_toBL,RfDf,confidence=confidence2,proportion=proportion2,tau=tau2,Lambda=Lambda2,historical_returns=historical_returns,modifiedlambda=modifiedlambda)
        #print(len(BLWeight))
        for index in range(len(BLWeight)):
            StockQty.iloc[i,index+2]=(BLWeight.iloc[index,0]*CurrentValue)/df.iloc[i,index+2] #qty = weight*total value/price
      else :
        spaceindays+=1
        for stocks in range(2,StockQty.shape[1]-1):
          StockQty.iloc[i,stocks]=StockQty.iloc[i-1,stocks] #same qty


      #value of pf

      GainOrLoss = 0
      for stocks in range(2, StockQty.shape[1]-1):
        qty = StockQty.iloc[i, stocks]

        if qty != 0.0:
            price_now = df.iloc[i, stocks]
            price_prev = df.iloc[i-1, stocks]
            GainOrLoss += qty * (price_now - price_prev)

      daily_rate = GetRiskFree(df, str(df.iloc[i,0]), 1, RfDf)
      interest_gain = (CurrentValue * RiskFreeAmount) * daily_rate
      CurrentValue += GainOrLoss + interest_gain - fees
      StockQty.iloc[i,-1]=CurrentValue


    StockQty = StockQty.iloc[start:].reset_index(drop=True)
    return StockQty
RfDf=GetRfDataframe(df)
final = Backtester(dfbacktest, hold=hold, hist=hist, proportion=proportion, df_toBL=df,RfDf=RfDf,confidence2=confidence,proportion2=proportion,tau2=tau,Lambda2=Lambda,start=500,modifiedlambda=0)

console.print("\n[green]✅ Backtest terminé avec succès ![/green]\n")

╭─────────────────────────╮
│ 📊 PORTFOLIO BACKTESTER │
│ Black-Litterman Model   │
╰─────────────────────────╯

⚙️  Configuration :

• Hold period: 1 mois

• Historique: 0 mois

• Proportion: 4

...

[(0.0004516817906780342, 0.9426727299358543, 1.4778832915476834e-15, 0), (0.0013937126238619995, 0.45205704905527905, 5.866339720483362e-16, 13), (0.001091659285711829, 1.2680532793169998, 3.4263817092686247e-16, 12), (-0.0017410298180955091, 0.47478557912983665, -9.954420210820425e-16, 3), (-0.00022287372008161264, 0.5538859990028783, -1.0567492668293675e-15, 8), (-0.0009607426264551228, 0.3619526369480509, -1.133618996257484e-15, 17), (0.00047104212893825696, 1.2525214191810687, -1.1894742325785838e-15, 6), (0.00019486774370468273, 0.5752338648651764, -1.4984978377269085e-15, 16), (0.0006260683302316303, 0.9990644091610992, -1.748719751658586e-15, 2), (0.0011345334932513103, 1.2555169965252075, -1.865176416949167e-15, 21), (-0.0004601768753799936, 1.3028287054104555, -2.227444245871963e-15, 11), (0.00043029330367856184, 1.0100204383704239, -2.230127707919464e-15, 20), (-0.0004461194125735888, 0.7383454833199472, -2.426239646094878e-15, 1), (-0.0023878184889340147, 2.0615521234206478,

Backtesting: 100%|█████████▉| 5273/5283 [01:07<00:00, 63.89it/s]

[(-0.0021180641562909114, 0.8210954736880832, 9.261303258823368e-15, 10), (-0.0037169639141579596, 1.5924335448438758, 7.290903781263327e-15, 22), (-0.0015476496925228005, 0.5330440002213197, 7.220202757565594e-15, 4), (-0.0012806247368622524, 1.112544352578928, 6.883418116302124e-15, 5), (-0.001867152784551914, 0.48359533058205767, 6.723748616671632e-15, 3), (-0.0007941827145920829, 0.9674803703653126, 4.8107387761653e-15, 14), (-0.000725506633578495, 0.40891993866613313, 3.7442150110068074e-15, 17), (-0.0007771267410440775, 1.1893839035052345, 3.0779614831357743e-15, 21), (-0.00013780075132079216, 0.11642125404931025, 1.7215939825986917e-15, 13), (-0.0005115434360732675, 0.26878988691586697, 1.5374700472726719e-15, 15), (-0.0001449020996843245, 0.362484709825259, 8.046587613764796e-16, 8), (3.900207312930667e-05, 0.5710659727381086, 7.666918050950377e-16, 1), (0.00030662270728529293, 1.1490108997871096, 4.662043614696874e-16, 19), (0.00032477253508002995, 1.1394515200286028, -5.54906

Backtesting: 100%|██████████| 5283/5283 [01:07<00:00, 77.78it/s]


✅ Backtest terminé avec succès !

In [95]:
import pandas as pd

money_norm = (final["Money"]/10000000*100) - 100
spx_norm = (final["SPX"]/final["SPX"].iloc[0]*100) - 100

df_plot = pd.DataFrame({
    "Date": final["Date"],
    "Portfolio": money_norm,
    "SPX": spx_norm
}).melt(id_vars="Date", var_name="Série", value_name="Évolution en %")

fix = px.line(
    df_plot,
    x="Date",
    y="Évolution en %",
    color="Série",
    color_discrete_map={"SPX": "red", "Portfolio": "green"},
    title="Comparaison des évolutions en %"
)

fix.update_layout(hovermode="x unified")
fix.show()


In [96]:
def calculate_historical_var_es(df, col_name='Money', confidence_level=0.95):


    returns = df[col_name].pct_change().dropna()

    cutoff = 1 - confidence_level

    var_value = returns.quantile(cutoff)

    worst_returns = returns[returns <= var_value]
    es_value = worst_returns.mean()

    return {
        "confidence_level": confidence_level,
        "VaR": -var_value,
        "ES": -es_value,
        "count_returns": len(returns),
        "count_breaches": len(worst_returns)
    }



def calculate_sharpe_ratio(df, col_name='close', risk_free_rate_annual=0.04):


    returns = (df[col_name] - df[col_name].shift(1)) / df[col_name].shift(1)
    returns = returns.dropna()
    rf_daily = risk_free_rate_annual / 252
    excess_returns = returns - rf_daily

    sharpe_daily = excess_returns.mean() / excess_returns.std()

    sharpe_annualized = sharpe_daily * np.sqrt(252)

    return sharpe_annualized


In [97]:
print("Portfolio Risk Measures:")
print(calculate_historical_var_es(final, 'Money', 0.99))
print(f"Sharpe Ratio: {calculate_sharpe_ratio(final, 'Money', 0.03):.2f}")

print("\nSPX Risk Measures:")
print(calculate_historical_var_es(final, 'SPX', 0.99))
print(f"Sharpe Ratio: {calculate_sharpe_ratio(final, 'SPX', 0.03):.2f}")

Portfolio Risk Measures:
{'confidence_level': 0.99, 'VaR': 0.042531502959310435, 'ES': 0.057602827675296134, 'count_returns': 5282, 'count_breaches': 53}
Sharpe Ratio: 0.30

SPX Risk Measures:
{'confidence_level': 0.99, 'VaR': 0.03443042477641993, 'ES': 0.050870785515521946, 'count_returns': 5282, 'count_breaches': 53}
Sharpe Ratio: 0.33


In [98]:
import pandas as pd
import numpy as np
import plotly.express as px

df2 = final[["Date", "SPX", "Money"]].copy()
df2["Portfolio"] = df2["Money"]
df2.drop(columns="Money", inplace=True)
df2["Date"] = pd.to_datetime(df2["Date"], dayfirst=True)
df2.set_index("Date", inplace=True)

daily_returns = df2.pct_change()
daily_returns.dropna(inplace=True)

daily_returns["Volatility of the Benchmark"] = daily_returns['SPX'].rolling(window=252).std() * np.sqrt(252)
daily_returns["Volatility of the Portfolio"] = daily_returns['Portfolio'].rolling(window=252).std() * np.sqrt(252)
data_to_plot = daily_returns.dropna()


fig = px.line(data_to_plot,
              x=data_to_plot.index,
              y=["Volatility of the Benchmark", "Volatility of the Portfolio"],
              labels={"value": "Volatility of the Benchmark", "variable": "Actif", "Date": "Date"},
              title="annualized vol : SPX vs Portfolio")

fig.show()

In [89]:
#multiple runs on variable start date :

numberofdays = 0
all_results = []

for i in range(numberofdays):
    current_start = 181 + i
    results_df = Backtester(dfbacktest, hold=hold, hist=hist, proportion=proportion,
                            df_toBL=df, RfDf=RfDf, confidence2=confidence,
                            proportion2=proportion, tau2=tau, Lambda2=Lambda,
                            start=current_start,modifiedlambda=0)

    money_norm = (results_df["Money"] / 10_000_000 * 100) - 100
    temp_df = pd.DataFrame({f"Iter_{i}": money_norm.values})

    dateresults = results_df["Date"]
    temp_df.index = dateresults

    all_results.append(temp_df)


global_df = pd.concat(all_results, axis=1)
global_df_clean = global_df.dropna()
print(global_df_clean.head())




ValueError: No objects to concatenate

In [23]:
global_df_clean

,Iter_0,Iter_1,Iter_2,Iter_3,Iter_4,Iter_5,Iter_6,Iter_7,Iter_8,Iter_9,...,Iter_11,Iter_12,Iter_13,Iter_14,Iter_15,Iter_16,Iter_17,Iter_18,Iter_19,Iter_20
Date,,,,,,,,,,,,,,,,,,,,,
2002-10-09,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2002-10-10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2002-10-11,2.954934,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2002-10-14,3.690886,0.635123,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2002-10-15,7.074880,4.184604,3.428509,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-02-23,700.258166,611.183934,632.759561,676.375633,744.047708,653.320313,796.280771,706.548068,544.072540,493.371567,...,533.570798,591.696342,712.919678,637.270149,788.753176,718.225388,553.331777,619.501151,613.029764,644.122724
2024-02-26,698.687244,610.004753,630.904152,674.991391,746.493154,655.817003,800.102256,708.330991,545.469718,494.818194,...,532.630962,589.137143,709.746590,634.868524,786.915906,716.484640,552.143911,618.512348,611.811376,642.741473
2024-02-27,700.653847,611.683283,632.493696,676.721164,749.947769,657.640369,802.947907,711.903607,548.085162,497.366447,...,535.375592,591.864875,713.262011,637.903149,789.117177,718.110372,553.681383,619.929942,613.347245,644.337473


In [24]:
#add spx to compare

dfcopyfinal = final[["Date", "SPX"]].copy()
dfcopyfinal.index=dfcopyfinal["Date"]
dfcopyfinal.drop(columns="Date", inplace=True)

global_df_clean = global_df_clean.merge(dfcopyfinal,left_index=True,right_index=True,how="left")
spx_norm = (global_df_clean["SPX"]/global_df_clean["SPX"].iloc[0]*100) - 100
global_df_clean["SPX"] = spx_norm

In [30]:
global_df_clean.tail(1)

,Iter_0,Iter_1,Iter_2,Iter_3,Iter_4,Iter_5,Iter_6,Iter_7,Iter_8,Iter_9,...,Iter_12,Iter_13,Iter_14,Iter_15,Iter_16,Iter_17,Iter_18,Iter_19,Iter_20,SPX
Date,,,,,,,,,,,,,,,,,,,,,
2024-02-29,699.430653,610.566525,630.711731,675.486037,747.957855,654.836892,799.745785,710.607345,546.858747,496.220984,...,590.498475,710.9466,635.819898,787.912861,717.29914,552.243658,618.492724,612.093025,642.848128,552.680365


In [27]:
import plotly.express as px

fig = px.line(
    global_df_clean,
    title=f"Comparaison des {numberofdays} itérations de Backtest (Rolling)",
    labels={
        "index": "Date",
        "value": "Performance Normalisée (%)",
        "variable": "Scénario"
    },
    template="plotly_dark"
)

fig.update_traces(line=dict(width=1.5))

if "SPX_Benchmark" in global_df_clean.columns:
    fig.update_traces(
        selector={"name": "SPX_Benchmark"},
        line=dict(width=4, color="white", dash="dot")
    )

fig.show()

In [158]:
alpha,beta,residus=Residual(GetReturnMonthly(df,"2018-05-11",21*36),GetReturnSPXMonthly(df,"2018-05-11",21*36))
fig = px.line(residus,
              x=residus.index,
              y=residus["S5BANKX"],
              title="quick draw of the evolution of the residual")

fig.show()

In [83]:
df_lambda = pd.DataFrame({
    "Lambda": StackLambda
})

fig = px.line(
    df_lambda,
    y="Lambda",
    title="Temporal evolution of λ",
    labels={"Lambda": "λ"}
)

fig.show()


In [84]:
import plotly.express as px
import pandas as pd

df_lambda = pd.DataFrame({"lambda": StackLambda})

fig = px.histogram(
    df_lambda,
    x="lambda",
    nbins=100,
    title="Lambda distribution",
    labels={"lambda": "λ"},
    opacity=0.8
)

fig.show()